# Single-objective Bayesian Optimization + compromise Decision Support Problem (cDSP)

This notebook accompanies the paper:

> H. M. Dilshad Alam Digonta, **Maryam Ghasemzadeh**, Anton van Beek, and Anand Balu Nellippallil.  
> *Design for ICME — A Data-Driven Decision Support Framework for Quantifying and Managing Uncertainty.*

It walks through the **mathematical example** of Section 4 in the paper: a 2-D quadratic interaction surface \(f(x_1,x_2)=(x_1+x_2)^2\) used to demonstrate that a cDSP-based Expected Improvement acquisition function concentrates samples in the **robust** region of the design space rather than only around the global optimum.

## Outline
1. Imports and helper functions (all defined in `main.py`)
2. Latin Hypercube initial design
3. The cDSP-based BO loop
4. Visualisation: sample distribution and convergence trace

## 1. Imports

We import everything from `main.py` so the notebook stays focused on **what** happens and the source stays focused on **how** it happens.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from main import (
    kernel,
    gp_regression,
    fit_hyperparameters,
    emi,
    deviation,
    expected_improvement,
    true_function,
    initial_design,
    build_test_grid,
    run_optimisation,
    plot_samples,
    plot_convergence,
)

np.random.seed(42)

## 2. Problem set-up

The toy objective is

$$
f(x_1,x_2) = x_1^{2} + 2\,x_1 x_2 + x_2^{2} = (x_1+x_2)^2.
$$

It is bowl-shaped with a degenerate minimum along the line \(x_1+x_2=0\). Inside the search domain \([0,100]^2\) the global minimum lies in the corner \((0,0)\); the *robust* region of interest is shifted away from that corner thanks to the cDSP formulation.

We start with 10 Latin Hypercube samples in \([0,10]^2\) and search on a 100 × 100 grid over \([0,100]^2\).

In [ ]:
x_sampled, y_sampled = initial_design(n_samples=10, lower=0.0, upper=10.0)
x_test = build_test_grid(lower=0.0, upper=100.0, n=100)

print('Initial X shape:', x_sampled.shape)
print('Initial Y shape:', y_sampled.shape)
print('Test grid shape:', x_test.shape)

## 3. cDSP-based BO loop

At every iteration we

1. Fit GP hyperparameters by Maximum-A-Posteriori (multi-start L-BFGS-B over a Latin Hypercube of initial $\omega$).
2. Predict the posterior $\mu(x),\,\sigma^2(x)$ on the candidate grid.
3. Convert $\mu$ to the **Error Margin Index** $\mathrm{EMI}(\mu,t,y_{\min}) = (\mu-t)/(\mu-y_{\min})$.
4. Form the cDSP deviation $d = 1 - \mathrm{EMI}/\mathrm{EMI}_{\text{target}}$ — small $d$ means *robust satisficing*.
5. Pick the next point by maximising Expected Improvement on $d$.
6. Evaluate the true objective there and append to the dataset.

Running 100 iterations takes a few minutes; reduce `n_iterations` if you just want to see the pipeline run.

In [ ]:
x_sampled, y_sampled = run_optimisation(
    n_iterations=100,
    target=10.0,
    emi_target=10.0,
    verbose=True,
)
print('Total samples after BO:', len(x_sampled))
print('Best observed y       :', float(np.min(y_sampled)))

## 4. Visualisation

### 4.1 Sample distribution in the design space
Blue circles are the initial LHS design; red crosses are points acquired by the cDSP-driven BO. We expect the acquired points to cluster in the **robust** region rather than only at the lower-left corner where the unconstrained minimum sits.

In [ ]:
plot_samples(x_sampled, n_initial=10, save_path='../figures/samples.png')
from IPython.display import Image
Image('../figures/samples.png')

### 4.2 Convergence trace
Best-so-far value of $y$ at each iteration.

In [ ]:
plot_convergence(y_sampled, save_path='../figures/convergence.png')
Image('../figures/convergence.png')

## 5. Citation
If you use this code, please cite the paper (see `CITATION.cff`).